# PatchCamelyon fixed-reference moving-window PCA extension 

This notebook adds an ungated moving-window PCA sensitivity experiment to the primary benchmark PatchCamelyon observation-scale ARL benchmark. It loads definitions from the primary notebook without running the primary experiment, and it reuses the completed seed-specific Phase I PCA objects and VAE checkpoints.

The run uses ResNet-18 layer 1, ResNet-18 layer 4, and the dataset-specific VAE latent representation. For every complete 50-image FIFO window, a new rank-20 PCA model is fitted and aligned to the frozen Phase I basis. The detectors are projection-subspace distance, T2, symmetric Gaussian KL divergence, Gaussian Hellinger distance, RBF MMD, and aligned MEWMA.


In [ ]:
from pathlib import Path
from dataclasses import replace
import json
import os

DATASET = "patchcamelyon"
QUICK_RUN = os.environ.get("MWPCA_QUICK", "0") == "1"
RUN_MWPCA = os.environ.get("MWPCA_RUN_FULL", "1") == "1"
SOURCE_NAME = "patchcamelyon_static_pca_benchmark.ipynb"
HELPER_NAME = "moving_window_pca_fixed_reference.py"
project_candidate = Path.home() / "Documents" / "Stats Thesis notebooks and results"
PROJECT_ROOT = next((root for root in (Path.cwd(), project_candidate)
                     if (root / SOURCE_NAME).is_file() and (root / HELPER_NAME).is_file()), Path.cwd())
SOURCE_NOTEBOOK = PROJECT_ROOT / SOURCE_NAME
HELPER_SCRIPT = PROJECT_ROOT / HELPER_NAME
if not SOURCE_NOTEBOOK.is_file() or not HELPER_SCRIPT.is_file():
    raise FileNotFoundError("Keep this notebook, the PatchCamelyon static-PCA notebook, and the MWPCA helper in the same folder")

os.environ["DRIFT_RUN_FULL"] = "0"
os.environ["DRIFT_QUICK"] = "1" if QUICK_RUN else "0"
os.environ["DRIFT_RUN_BASE"] = str(PROJECT_ROOT)
document = json.loads(SOURCE_NOTEBOOK.read_text())
for cell_index, cell in enumerate(document["cells"]):
    if cell_index >= 24:
        break
    if cell.get("cell_type") != "code":
        continue
    source = "".join(cell.get("source", []))
    source = "\n".join(line for line in source.splitlines() if not line.lstrip().startswith("%"))
    exec(compile(source, f"{SOURCE_NOTEBOOK.name}:cell{cell_index}", "exec"), globals())

CFG = replace(CFG, seeds=tuple(range(42, 62)), mc_arl1_reps=20)
print(f"Loaded {DATASET} pipeline from {SOURCE_NOTEBOOK}")


In [ ]:
exec(compile(HELPER_SCRIPT.read_text(), str(HELPER_SCRIPT), "exec"), globals())
MWPCA_CFG = replace(MovingWindowPCAConfig.from_base(CFG), run_full=RUN_MWPCA)
display(mwpca_design_audit(CFG, MWPCA_CFG))
print(f"output root: {MWPCA_CFG.out_root}")
print(f"seeds: {CFG.seeds}")
print(f"drift episodes per exact condition per seed: {CFG.mc_arl1_reps}")
print(f"ARL0 calibration/held-out episodes: {MWPCA_CFG.calibration_episodes}/{MWPCA_CFG.heldout_episodes}")


## Detector and timing notes

The first score is available at observation 50 and contains 49 in-control observations plus the first drifted observation. The PCA window advances by one image and is refitted at every score. All detectors receive a fresh observation-scale ARL0 calibration targeting 370 observations, followed by a disjoint held-out in-control audit. Because this ungated MWPCA is a finite 50-observation functional, dependent moving-block resampling of its validation score and innovation banks remains consistent with the primary design. MEWMA is replayed from zero state inside every bootstrap episode.

Local in-sample SPE is not included as a detector. A PCA fit minimizes the residual energy of the same window being scored, so that quantity is not the static SPE chart used in the primary benchmark. Projection-subspace distance is the direct replacement. The local axes are Procrustes-aligned to the Phase I axes before T2, KL, Hellinger, MMD, and MEWMA are calculated. Because H=50 is small relative to k=20 under classical MWPCA rules of thumb, covariance-based statistics use Ledoit-Wolf shrinkage and empirical ARL calibration rather than asymptotic T2 or Q limits.

The pilot uses 10 seed-repartitioned experiments and one paired test episode per exact pattern/mechanism/severity condition. This is enough to check feasibility and gross behavior, but its condition-level uncertainty should not be treated as final evidence. Outputs are written condition by condition and are resumable.


In [ ]:
if MWPCA_CFG.run_full:
    MWPCA_RESULTS = run_mwpca_extension(CFG, MWPCA_CFG, EXTRACTOR)
    display(MWPCA_RESULTS["heldout_summary"])
    display(MWPCA_RESULTS["summary"].head(30))
else:
    print("Configuration validated. Set RUN_MWPCA=True or MWPCA_RUN_FULL=1 to execute.")
